# Experimentos con nuestra reimplementación de FlowNP

Cada persona elige una **TAREA** en la celda 1 y ejecuta las celdas 1 a 4. Todo se guarda en la carpeta
compartida de Drive (`flownp_runs/ours/`) y, si Colab se desconecta, basta con volver a ejecutar las celdas
1 a 4: cada ejecución sigue desde su último punto de control y las terminadas se saltan.

| TAREA | Qué hace | Para qué | Tiempo aprox. en T4 |
|---|---|---|---|
| `"h1"` | Entrena **nuestro FlowNP** en el núcleo exponencial cuadrático, 5 semillas, y lo evalúa sobre el mismo conjunto que el oficial | Hipótesis H1: ¿da lo mismo que el oficial (1,716 ± 0,004)? | ~4 h |
| `"cosenos"` | Entrena nuestro FlowNP y los modelos oficiales (condicional, atento, autorregresivo y de difusión) sobre los cosenos partiendo de sus puntos de control del núcleo exponencial cuadrático, más FlowNP desde cero; 3 semillas; los compara con el posterior exacto | Fidelidad y diversidad; casos con varias respuestas posibles | ~2,5 h por semilla: repartid las semillas entre cuentas con `SEEDS` |
| `"evaluar"` | No entrena: carga los puntos de control **oficiales** ya guardados y mide calibración, y el barrido de resolvedores de FlowNP | Calibración (H2) y coste del muestreo | ~2–3 h |
| `"difusion"` | Entrena el modelo de difusión oficial en el núcleo exponencial cuadrático, 3 semillas, con el código oficial | Comparación FlowNP frente a difusión con guiado | ~3 h |

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4*.

## 1. Configuración — elige la TAREA

In [ ]:
TAREA = "h1"
SEEDS = None
PARALLEL = 2
POLL = 60
NUM_STEPS = 100_000

REPO = "https://github.com/samuel13032004/flownp-repro.git"
OFFICIAL_COMMIT = "f3403b6479dad33a7b93ba6edcf93c53d40f8e65"
DRIVE_DIR = "/content/drive/MyDrive/flownp_runs"
OFFICIAL_OUT = f"{DRIVE_DIR}/official"
OUT = f"{DRIVE_DIR}/ours"
CODE = "/content/flownp-repro"
OFFICIAL_CODE = "/content/third_party/flowNP"
print("Tarea:", TAREA, "| salida:", OUT)

## 2. Preparar el entorno

In [ ]:
import os, subprocess, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
assert gpu, "No hay tarjeta gráfica: Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> T4"
print("Tarjeta gráfica:", gpu)
if DRIVE_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(OUT, exist_ok=True)

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError(cmd)
    return r.stdout.strip()

if os.path.isdir(CODE):
    sh(f"git -C {CODE} pull -q")
else:
    sh(f"git clone -q {REPO} {CODE}")
print("Nuestro código en el commit", sh(f"git -C {CODE} log -1 --format='%h %s'"))
if not os.path.isdir(OFFICIAL_CODE):
    sh(f"git clone -q https://github.com/danrsm/flowNP.git {OFFICIAL_CODE}")
sh(f"git -C {OFFICIAL_CODE} checkout -q {OFFICIAL_COMMIT}")
sh(f"{sys.executable} -m pip -q install -e {CODE} attrdictionary addict prettytable torchdiffeq pyyaml")
sh(f"{sys.executable} -m pip -q install --no-deps flow-matching==1.0.10")
sys.path.insert(0, f"{CODE}/src")
open(f"{OFFICIAL_CODE}/utils/paths.py", "w").write(f'''import os
datasets_path = "{OFFICIAL_OUT}/datasets"
output_path = "{OFFICIAL_OUT}"
evalsets_path = os.path.join(output_path, "evalsets")
results_path = os.path.join(output_path, "results")
''')
print(sh(f"cd {CODE} && {sys.executable} -m pytest -q tests 2>&1 | tail -1"), "(pruebas de nuestro código)")

## 3. Lista de trabajos de la tarea

Cada trabajo es: entrenar (si hace falta) y después evaluar, guardando las medidas en `metrics.json`.

In [ ]:
import os, re, json, glob, torch
PY = sys.executable

def ckpt_step(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)["step"]
    except Exception:
        return None

def has_keys(path, keys):
    try:
        d = json.load(open(path))
        return all(k in d for k in keys)
    except Exception:
        return False

def last_step_of(log_glob):
    logs = sorted(glob.glob(log_glob))
    steps = re.findall(r"step (\d+) ", open(logs[-1]).read()) if logs else []
    return int(steps[-1]) if steps else 0

def ours_job(name, run, model, data, seed, eval_tasks, evalset, keys, init=None):
    train = (f"{PY} scripts/train.py --run {run} --model {model} --data {data} --seed {seed} "
             f"--steps {NUM_STEPS} --official-code {OFFICIAL_CODE}" + (f" --init {init}" if init else ""))
    ev_model = model if model == "flownp" else model
    evaluate = (f"{PY} scripts/evaluate.py --ckpt {run}/ckpt.pt --model {ev_model} --evalset {evalset} "
                f"--tasks {eval_tasks} --out {run}/metrics.json --official-code {OFFICIAL_CODE}")
    return dict(name=name, cmd=lambda: f"{train} && {evaluate}",
                done=lambda: has_keys(f"{run}/metrics.json", keys),
                progress=lambda: f"{last_step_of(f'{run}/train.log')}")

def eval_job(name, ckpt, model, evalset, tasks, out, keys):
    cmd = (f"{PY} scripts/evaluate.py --ckpt {ckpt} --model {model} --evalset {evalset} --tasks {tasks} "
           f"--out {out} --official-code {OFFICIAL_CODE}")
    return dict(name=name, cmd=lambda: cmd, done=lambda: has_keys(out, keys), progress=lambda: "evaluando")

def official_train_job(kernel, model, seed):
    d = f"{OFFICIAL_OUT}/results/gp/main/{kernel}/{model}/s{seed}"
    def state():
        st = ckpt_step(f"{d}/ckpt.tar")
        ev = f"{d}/eval_{kernel}.log"
        evaluated = os.path.isfile(ev) and "tar_ll" in open(ev).read()
        if st is None:
            return "nueva"
        if st > NUM_STEPS:
            return "terminada" if evaluated else "evaluar"
        return "reanudar"
    def cmd():
        s = state()
        if s == "nueva" and os.path.isfile(f"{d}/ckpt.tar"):
            os.remove(f"{d}/ckpt.tar")
        c = (f"cd {OFFICIAL_CODE} && {PY} gp.py --mode={'eval' if s == 'evaluar' else 'train'} --kernel={kernel} "
             f"--model={model} --expname=main --expid=s{seed} --train_seed={seed} --num_steps={NUM_STEPS} "
             f"--print_freq=1000 --save_freq=1000 --eval_freq=1000000000 --plot_batch_size=4 --plot_num_samples=10")
        return c + (" --resume=1" if s == "reanudar" else "")
    return dict(name=f"oficial_{kernel}_{model}_s{seed}", cmd=cmd, done=lambda: state() == "terminada",
                progress=lambda: f"{last_step_of(f'{d}/train_*.log')}")

JOBS = []
if TAREA == "h1":
    ev = f"{OFFICIAL_OUT}/evalsets/gp/rbf-seed0.tar"
    assert os.path.isfile(ev), f"Falta {ev}: es el conjunto de evaluación oficial (lo generó el cuaderno de referencias)"
    for s in (SEEDS if SEEDS is not None else [0, 1, 2, 3, 4]):
        JOBS.append(ours_job(f"h1_flownp_rbf_s{s}", f"{OUT}/h1/flownp/rbf/s{s}", "flownp", "gp:rbf", s,
                             "likelihood,calibration", ev, ["likelihood", "calibration"]))

elif TAREA == "cosenos":
    from flownp.data import evalsets
    from flownp.data.cosine import CosineSampler
    from flownp.evaluation import exact_cosine_likelihood
    ev = f"{OUT}/evalsets/cosine-seed0.pt"
    if not os.path.isfile(ev):
        os.makedirs(os.path.dirname(ev), exist_ok=True)
        batches = evalsets.make_evalset(CosineSampler(), num_batches=50, batch_size=128, seed=0)
        evalsets.save(batches, ev)
        json.dump({"tar_ll": exact_cosine_likelihood(CosineSampler(), batches)}, open(f"{OUT}/evalsets/cosine-exact.json", "w"))
    print("Posterior exacto sobre el conjunto de evaluación:", json.load(open(f"{OUT}/evalsets/cosine-exact.json")))
    def rbf_ckpt(m, s):
        ck = f"{OFFICIAL_OUT}/results/gp/main/rbf/{m}/s{s}/ckpt.tar"
        return ck if os.path.isfile(ck) and (ckpt_step(ck) or 0) > NUM_STEPS else None
    tasks, keys = "likelihood,calibration,cosine", ["likelihood", "calibration", "cosine"]
    for s in (SEEDS if SEEDS is not None else [0, 1, 2]):
        for name, model, off in [("flownp", "flownp", "fnp"), ("tnpa", "official:tnpa", "tnpa"),
                                 ("cnp", "official:cnp", "cnp"), ("anp", "official:anp", "anp"),
                                 ("ndp", "official:ndp", "ndp")]:
            init = rbf_ckpt(off, s)
            if init is None:
                print(f"  (se omite {name} semilla {s}: falta su punto de control oficial terminado en el núcleo rbf)")
                continue
            JOBS.append(ours_job(f"cos_{name}_s{s}", f"{OUT}/cosine/{name}/s{s}", model, "cosine", s,
                                 tasks, ev, keys, init=init))
        JOBS.append(ours_job(f"cos_flownp_desde_cero_s{s}", f"{OUT}/cosine/flownp_desde_cero/s{s}", "flownp",
                             "cosine", s, tasks, ev, keys))

elif TAREA == "evaluar":
    seeds = SEEDS if SEEDS is not None else [0, 1, 2, 3, 4]
    for k in ["rbf", "periodic", "matern"]:
        ev = f"{OFFICIAL_OUT}/evalsets/gp/{k}-seed0.tar"
        for m in ["fnp", "cnp", "anp", "tnpa", "ndp"]:
            for s in seeds:
                ck = f"{OFFICIAL_OUT}/results/gp/main/{k}/{m}/s{s}/ckpt.tar"
                if not (os.path.isfile(ck) and os.path.isfile(ev) and (ckpt_step(ck) or 0) > NUM_STEPS):
                    continue
                out = f"{OUT}/eval_official/{k}/{m}/s{s}"
                if m == "fnp":
                    sweep = s == 0 or (k == "rbf" and s in (1, 2))
                    tasks = "calibration,solvers" if sweep else "calibration"
                    JOBS.append(eval_job(f"eval_{k}_fnp_s{s}", ck, "official-fnp", ev, tasks, f"{out}/metrics.json",
                                         tasks.split(",")))
                    JOBS.append(eval_job(f"eval_{k}_fnpestoc_s{s}", ck, "official:fnp", ev, "calibration",
                                         f"{out}/metrics_estocastico.json", ["calibration"]))
                else:
                    JOBS.append(eval_job(f"eval_{k}_{m}_s{s}", ck, f"official:{m}", ev, "calibration",
                                         f"{out}/metrics.json", ["calibration"]))

elif TAREA == "difusion":
    for s in (SEEDS if SEEDS is not None else [0, 1, 2]):
        JOBS.append(official_train_job("rbf", "ndp", s))
else:
    raise ValueError("TAREA debe ser h1, cosenos, evaluar o difusion")

pend = [j for j in JOBS if not j["done"]()]
print(f"{len(JOBS)} trabajos, {len(JOBS) - len(pend)} ya terminados, {len(pend)} por hacer")
for j in pend[:8]:
    print("  ", j["name"])
if len(pend) > 8:
    print("   ...")

## 4. Lanzar

Deja esta celda corriendo con la pestaña abierta. Informa cada minuto. Los registros completos de cada
trabajo quedan en `OUT/logs/`.

In [ ]:
import subprocess, time

def launch_all():
    pending = [j for j in JOBS if not j["done"]()]
    print(f"{len(pending)} trabajos por hacer")
    env = dict(os.environ, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
    running, tries, t0 = {}, {}, time.time()
    os.makedirs(f"{OUT}/logs", exist_ok=True)
    while pending or running:
        while pending and len(running) < PARALLEL:
            j = pending.pop(0)
            log = open(f"{OUT}/logs/{j['name']}.out", "a")
            running[j["name"]] = (j, subprocess.Popen(j["cmd"](), shell=True, cwd=CODE, stdout=log,
                                                      stderr=subprocess.STDOUT, env=env))
            print(f"[{time.strftime('%H:%M')}] lanza {j['name']}", flush=True)
        time.sleep(POLL)
        for name, (j, p) in list(running.items()):
            if p.poll() is not None:
                del running[name]
                ok = j["done"]()
                tries[name] = tries.get(name, 0) + (0 if ok else 1)
                retry = not ok and tries[name] <= 2
                if retry:
                    pending.append(j)
                print(f"[{time.strftime('%H:%M')}] {'OK   ' if ok else 'FALLO'} {name}"
                      + ("" if ok else f" (código {p.returncode}{', se reintenta' if retry else ''}) -> mira {OUT}/logs/{name}.out"),
                      flush=True)
        if running:
            prog = ", ".join(f"{n} {j['progress']()}" for n, (j, _) in running.items())
            print(f"   {(time.time() - t0) / 60:5.1f} min · en marcha: {prog}", flush=True)
    print("Hecho.")

launch_all()

## 5. Resultados

Solo necesita las celdas 1 y 2. Muestra un resumen de todo lo que haya en `OUT` y lo guarda en CSV.

In [ ]:
import json, glob, os, statistics as st
def ms(v):
    return (st.mean(v), st.stdev(v) if len(v) > 1 else float("nan"), len(v))
def show(title, groups, fields):
    print(f"\n=== {title}")
    for g, runs in sorted(groups.items()):
        parts = []
        for label, getter in fields:
            vals = [x for x in (getter(r) for r in runs) if x is not None]
            if vals:
                m, s, n = ms(vals)
                parts.append(f"{label} {m:.4f} ± {s:.4f}")
        if parts:
            print(f"  {g:28s} (n={len(runs)})  " + " · ".join(parts))
def get(r, *path):
    for p in path:
        if not isinstance(r, dict) or p not in r:
            return None
        r = r[p]
    return r
rows = []
for f in sorted(glob.glob(f"{OUT}/**/metrics*.json", recursive=True)):
    d = json.load(open(f)); d["_file"] = os.path.relpath(f, OUT); rows.append(d)
    parts = d["_file"].split("/")
    d["_group"] = "/".join(parts[:-2]) + ("/estocástico" if "estocastico" in parts[-1] else "")
groups = {}
for r in rows:
    groups.setdefault(r["_group"], []).append(r)
show("Verosimilitud y calibración", groups, [
    ("verosimilitud", lambda r: get(r, "likelihood", "tar_ll")),
    ("CRPS", lambda r: get(r, "calibration", "crps")),
    ("cobertura 50 %", lambda r: get(r, "calibration", "coverage_50")),
    ("cobertura 90 %", lambda r: get(r, "calibration", "coverage_90")),
])
ex = f"{OUT}/evalsets/cosine-exact.json"
if os.path.isfile(ex):
    print("\nPosterior exacto de los cosenos, verosimilitud:", round(json.load(open(ex))["tar_ll"], 4))
cos = {g: rs for g, rs in groups.items() if any("cosine" in r for r in rs)}
for M in ["1", "2", "3", "5", "10", "20"]:
    show(f"Cosenos con {M} puntos de contexto", cos, [
        ("son coseno", lambda r: get(r, "cosine", M, "fidelity_is_cosine")),
        ("dist. frecuencias", lambda r: get(r, "cosine", M, "freq_tv_distance")),
        ("modos hallados", lambda r: get(r, "cosine", M, "plausible_modes_found")),
        ("densidad", lambda r: get(r, "cosine", M, "density")),
        ("cobertura", lambda r: get(r, "cosine", M, "coverage")),
    ])
sw = [r for r in rows if "solvers" in r]
if sw:
    print("\n=== Barrido de resolvedores (FlowNP oficial)")
    for r in sw:
        print(" ", r["_file"])
        for x in r["solvers"]:
            print(f"     {x['method']:8s} {x['steps']:3d} pasos: verosimilitud {x['tar_ll']:.4f}  CRPS {x['crps']:.4f}  "
                  f"cobertura 90 % {x['coverage_90']:.3f}  muestreo {x['sample_seconds']:.1f} s")
json.dump(rows, open(f"{OUT}/resumen.json", "w"), indent=1)
print("\nTodo guardado en", f"{OUT}/resumen.json")